In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.signal import find_peaks, peak_widths, savgol_filter
import pandas as pd
import h5py
from typing import Literal
import struct

plt.rcParams["figure.dpi"] = 256

In [ ]:
def plot_with_peaks(
    data,
    n_bins,
    prominence=None,
    xlabel="",
    title="",
    xlims=None,
    savgol_window=None,
    savgol_strength=1.0,
    clip_counts=None,
    stacked=False,
    legend=None,
    bin_range=None,
    text_size=5,
):
    if not stacked:
        signals = [data]
    else:
        signals = data

    bottom = np.zeros(n_bins)

    for signal in signals:
        counts, bins = np.histogram(signal, bins=n_bins, density=False, range=bin_range)
        if savgol_window is not None:
            background = savgol_filter(counts, window_length=savgol_window, polyorder=3)
            counts = counts - background * savgol_strength

        if clip_counts is not None:
            counts[counts > clip_counts] = clip_counts

        bin_width = np.diff(bins).mean()
        bin_centers = (bins[:-1] + bins[1:]) / 2.0
        height_scale = counts.max()

        plt.bar(bin_centers, counts, width=bin_width, align="center", bottom=bottom)
        plt.xlim(xlims)
        bottom += counts
        bottom = np.round(bottom, 10)
        if prominence is not None:
            peaks, props = find_peaks(counts, prominence=prominence)
            half_widths = peak_widths(counts, peaks)[0] * bin_width
            for x, y, half_width in zip(bin_centers[peaks], counts[peaks], half_widths):
                plt.errorbar(
                    [x],
                    [y],
                    xerr=[half_width],
                    c="black",
                    fmt="",
                    capsize=2,
                    elinewidth=0.2,
                    capthick=0.2,
                )
                plt.annotate(
                    f"{x:.03f} ± {half_width:.03f} MeV",
                    (x, y + height_scale * (0.005 * text_size)),
                    ha="left",
                    fontsize=text_size,
                )

    plt.xlabel(xlabel)
    plt.ylabel("Count")
    # plt.ylim(ylims)
    plt.title(title)
    if legend is not None:
        plt.legend(legend)

    # plt.figure()
    plt.show()

# Loaders

In [ ]:
keys = [
    "E",
    "name",
    "z",
    "rho",
    "phi",
    "momentumZ",
    "momentumRho",
    "momentumPhi",
    "sourceDepth",
    "sourceRadius",
    "sourceE",
    "sourcePDG"
]

def decode(b):
    return b.decode("utf-8")


decode_v = np.vectorize(decode)


def load_hdf5(path):
    dfs = []
    with h5py.File(path, "r") as f:
        for key in keys:
            data = f["default_ntuples"]["Measurement"][key]["pages"][()]
            if isinstance(data[0], bytes):
                data = decode_v(data)

            dfs.append(pd.DataFrame(data, columns=[key]))

    return pd.concat(dfs, axis=1)


def load_csv(path):
    df = pd.read_csv(path, comment="#", header=None)
    df.columns = keys.copy()
    print(f"{df.shape[0]:,} interactions recorded")
    return df


def load_raw(path):
    rows = []
    with open(path, "rb") as file:
        while data := file.read(3 * 8):
            energy, p_z, depth = struct.unpack("ddd", data)
            rows.append([energy, p_z, depth])
    df = pd.DataFrame(rows, columns=["E", "p_z", "z"])
    return df

In [ ]:
df = load_csv("archive/predetection-2B-neutrons-only.csv")
df

# Analysis

In [ ]:
particle_names = np.unique(df["name"])
particle_nums = np.arange(0, len(particle_names))
particle_dict = dict(zip(particle_names,particle_nums))

## Masking Utilities

In [ ]:
property_range = lambda property, lower, upper, mask=None: (
    df if mask is None else df[mask]
)[property].between(lower, upper)
energy_range = lambda lower, upper, mask=None: property_range("E", lower, upper, mask)

In [ ]:
# plot_with_peaks(df["sourceDepth"], 100)
# plot_with_peaks(df["sourceRadius"], 100)
# plt.yscale("log")
# plot_with_peaks(df["z"], 100)
# plot_with_peaks(df["rho"], 100)
# plot_with_peaks(df["phi"], 100)
# plot_with_peaks(df["momentumZ"], 100)
# plot_with_peaks(df["momentumRho"], 100)
mask = df["z"].between(200, 368)
diff = (df["phi"]-df["momentumPhi"]) % (2 * np.pi) - np.pi
plot_with_peaks(diff, 2048, xlabel="Momentum Angle from Radial")
df[mask]

In [ ]:
plt.close()
mask = df["z"].between(0, 360)
plt.hexbin(
    df[mask]["rho"] * np.cos(df[mask]["phi"]),
    df[mask]["rho"] * np.sin(df[mask]["phi"]),
    # C=df["E"],
    # reduce_C_function=np.sum,
    linewidths=0.2,
    # vmax=0.5,
    norm=LogNorm(),
)
plt.colorbar(label="Count")
# plt.xlabel("$z$-Momentum Bias, $p_z$")
# plt.ylabel("Energy, $E$ (MeV)")
# plt.title("Number of Gamma Rays Produced Below 10 MeV")
plt.show()

plt.hexbin(
    df[mask]["rho"],
    df[mask]["z"],
    # C=df["E"],
    # reduce_C_function=np.sum,
    linewidths=0.2,
    # vmax=0.5,
    norm=LogNorm(),
)
plt.colorbar(label="Count")
# plt.xlabel("$z$-Momentum Bias, $p_z$")
# plt.ylabel("Energy, $E$ (MeV)")
# plt.title("Number of Gamma Rays Produced Below 10 MeV")
plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')

mask = ((df["momentumPhi"] - df["phi"]) % (2 * np.pi)).abs() < np.pi/2
d = df

# ax.quiver(
#     d["rho"] * np.cos(d["phi"]),
#     d["rho"] * np.sin(d["phi"]),
#     d["z"],
#     d["momentumRho"] * np.cos(d["momentumPhi"]),
#     d["momentumRho"] * np.sin(d["momentumPhi"]),
#     d["momentumZ"],
#     length=50,
#     linewidth=0.1,
#     arrow_length_ratio=0.3
# )
scatter_plot = ax.scatter(
    d["rho"] * np.cos(d["phi"]),
    d["rho"] * np.sin(d["phi"]),
    d["z"],
    s=0.01,
    c=d["name"].map(particle_dict),#d["sourceDepth"],#(d["phi"]-d["momentumPhi"]) % (2.0 * np.pi) - np.pi,
    depthshade=False,
    alpha=0.8,
    linewidths=0.1
)

ax.set_title("Transported Particles")

fig.colorbar(scatter_plot, ax=ax)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')

plt.show()